# 03 — Model Comparison & Final Selection

**Primary metric: PR-AUC** — at ~0.1% fraud rate, ROC-AUC and accuracy are misleading. PR-AUC measures how well we rank fraud cases among flagged transactions.

**Threshold:** Cost-optimized (FP=$50, FN=$500), not default 0.5.

**CV:** 5-fold stratified, report mean ± std.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

from src.models.baseline import train_baseline
from src.models.tree_ensemble import train_tree_ensemble
from src.models.anomaly_detection import train_anomaly_models
from src.evaluate import results_to_markdown_table
from src.utils import load_config, setup_logging

setup_logging()
config = load_config(PROJECT_ROOT / "config" / "config.yaml")

In [ ]:
_, lr_result = train_baseline(config)
tree_results = train_tree_ensemble(config)
anomaly_results = train_anomaly_models(config)

all_results = [lr_result] + [v["result"] for v in tree_results.values()] + list(anomaly_results.values())
print(results_to_markdown_table(all_results))

## Final Model Selection

After running on the full PaySim dataset, document tradeoffs explicitly:

> **Example template:** XGBoost chosen over Random Forest: +X% recall at comparable precision, acceptable latency for near-real-time scoring. Logistic regression retained as interpretable baseline. Isolation Forest useful when labels are scarce but underperforms on PR-AUC when labeled fraud data is available.

Compare imbalance strategies via `src/imbalance_strategies.py` — report which wins (class weight vs SMOTE vs SMOTE-ENN vs undersampling) and **why**, not just apply SMOTE blindly.

In [ ]:
metrics_dir = PROJECT_ROOT / config["paths"]["metrics_dir"]
rows = []
for path in sorted(metrics_dir.glob("*_metrics.json")):
    with path.open() as f:
        rows.append(json.load(f))
pd.DataFrame(rows)